# EmotionSense AI — RC1 Validation (Google Colab)

Validate the **frozen RC1** (`v1.0.0-rc1`) on real corpora. **No source code is changed.**
Colab has internet, so datasets are downloaded by the prepared `scripts/fetch_datasets.py`.

- RAVDESS downloads directly from Zenodo (no credentials).
- TESS + CREMA-D download via the Kaggle API (upload your `kaggle.json` in step 2).
- Runtime → Change runtime type → **GPU** optional (SSL extraction is CPU-bound in RC1).

## 1. Install the frozen RC1 release

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/<your-username>/EmotionSenseAI.git'  # <-- SET THIS
GIT_REF  = 'v1.0.0-rc1'  # frozen RC1 tag
if not os.path.isdir('EmotionSenseAI'):
    subprocess.run(['git','clone','--branch',GIT_REF,'--depth','1',REPO_URL], check=True)
os.chdir('EmotionSenseAI')
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[transformer,backend]'], check=True)
print('installed at', subprocess.run(['git','describe','--tags'],capture_output=True,text=True).stdout.strip())

## 2. Configure Kaggle credentials (for TESS + CREMA-D)

Get `kaggle.json` from Kaggle → Account → *Create New API Token*, then run and upload it.

In [ ]:
from google.colab import files
import os, shutil
print('Upload kaggle.json ...')
files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
subprocess.run([sys.executable,'-m','pip','install','-q','kaggle'], check=True)
print('kaggle configured')

## 3. Download the corpora into `data/raw/`

In [ ]:
!python scripts/fetch_datasets.py --all

## 4. CPU classical validation (primary acceptance run)

In [ ]:
# CPU classical validation: baselines + SVM/RF/LogReg, RAVDESS 5-fold + cross-corpus CREMA-D
!python scripts/run_benchmark.py --experiment configs/experiments/rc1_validation.yaml --out-dir experiments/reports

## 5. CREMA-D in-corpus CV

In [ ]:
# CREMA-D in-corpus 5-fold (91 speakers)
!python scripts/run_benchmark.py --experiment configs/experiments/rc1_cremad.yaml --out-dir experiments/reports

## 6. Transformer validation (Distil-HuBERT)

SSL extraction is CPU-bound in RC1; the CREMA-D cross-corpus pass dominates runtime.

In [ ]:
# Transformer: frozen Distil-HuBERT embeddings + SVM head (SSL extraction is CPU-bound in RC1).
# Embeddings are cached per clip; the CREMA-D cross-corpus pass (~7.4k clips) dominates runtime.
!python scripts/run_benchmark.py --experiment configs/experiments/rc1_transformer.yaml --out-dir experiments/reports

## 7. Results — download the leaderboards

In [ ]:
import glob
from google.colab import files
for md_file in sorted(glob.glob('experiments/reports/*.md')):
    print('='*80); print(md_file); print(open(md_file, encoding='utf-8').read())
for f in sorted(glob.glob('experiments/reports/*.json')):
    files.download(f)